# Notebook 05 — Restaurant Song Recommender

Takes a restaurant's PCA vector, maps it into Spotify PCA space using a weight matrix, then returns the closest matching songs via cosine similarity.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\processed"

songs         = pd.read_csv(os.path.join(PROCESSED, 'song_pca.csv'))
restaurants   = pd.read_csv(os.path.join(PROCESSED, 'restaurant_pca.csv'))

print(f"Songs:       {songs.shape}")
print(f"Restaurants: {restaurants.shape}")

Songs:       (1201245, 9)
Restaurants: (34808, 15)


---
## Step 1 — Separate identifiers from PC scores

In [2]:
SPOTIFY_PC_COLS = [c for c in songs.columns        if c.startswith('pc')]
YELP_PC_COLS    = [c for c in restaurants.columns  if c.startswith('pc')]

song_vecs       = songs[SPOTIFY_PC_COLS].values          # (1.2M, 6)
restaurant_vecs = restaurants[YELP_PC_COLS].values       # (52K, 13)

print(f"Song vectors:  {song_vecs.shape}")
print(f"Restaurant vectors: {restaurant_vecs.shape}")
print(f"\nSpotify PCs: {SPOTIFY_PC_COLS}")
print(f"Yelp PCs:    {YELP_PC_COLS}")

Song vectors:  (1201245, 6)
Restaurant vectors: (34808, 13)

Spotify PCs: ['pc1', 'pc2', 'pc3', 'pc4', 'pc5', 'pc6']
Yelp PCs:    ['pc1', 'pc2', 'pc3', 'pc4', 'pc5', 'pc6', 'pc7', 'pc8', 'pc9', 'pc10', 'pc11', 'pc12', 'pc13']


---
## Step 2 — Read the PCA loadings

Before writing the weight matrix we need to know what each PC actually represents.
The loadings tell us which original features drive each component.

In [3]:
pca_spotify = joblib.load(os.path.join(PROCESSED, 'spotify_pca.pkl'))
pca_yelp    = joblib.load(os.path.join(PROCESSED, 'yelp_pca.pkl'))

AUDIO_FEATURES = [
    'danceability', 'energy', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'loudness', 'tempo'
]

YELP_FEATURES = [
    'Ambience.romantic', 'Ambience.divey', 'Ambience.classy',
    'Ambience.hipster', 'Ambience.trendy', 'Ambience.upscale', 'Ambience.casual',
    'HasTV', 'HappyHour', 'RestaurantsGoodForGroups',
    'GoodForMeal.breakfast', 'GoodForMeal.brunch',
    'GoodForMeal.latenight', 'GoodForMeal.dinner',
    'RestaurantsTableService', 'NoiseLevel', 'stars'
]

# Trim to however many features survived after dropna / low-variance removal
n_yelp_feats    = pca_yelp.n_features_in_
n_spotify_feats = pca_spotify.n_features_in_
YELP_FEATURES   = YELP_FEATURES[:n_yelp_feats]
AUDIO_FEATURES  = AUDIO_FEATURES[:n_spotify_feats]

spotify_loadings = pd.DataFrame(
    pca_spotify.components_,
    index=SPOTIFY_PC_COLS,
    columns=AUDIO_FEATURES
).round(2)

yelp_loadings = pd.DataFrame(
    pca_yelp.components_,
    index=YELP_PC_COLS,
    columns=YELP_FEATURES
).round(2)

print("=== SPOTIFY loadings ===")
display(spotify_loadings)

print("\n=== YELP loadings ===")
display(yelp_loadings)

=== SPOTIFY loadings ===


,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,loudness,tempo
pc1,0.31,0.48,0.14,-0.43,-0.28,0.13,0.34,0.48,0.19
pc2,0.54,-0.29,0.36,0.33,-0.31,-0.11,0.39,-0.17,-0.33
pc3,-0.24,0.03,0.56,0.03,-0.12,0.73,-0.24,-0.06,-0.15
pc4,0.01,-0.15,0.24,0.22,0.05,0.05,0.18,-0.19,0.89
pc5,0.14,0.15,0.60,-0.24,0.61,-0.38,-0.15,-0.04,-0.07
pc6,0.20,0.06,-0.27,0.07,0.63,0.49,0.46,-0.09,-0.12



=== YELP loadings ===


,Ambience.romantic,Ambience.divey,Ambience.classy,Ambience.hipster,Ambience.trendy,Ambience.upscale,Ambience.casual,HasTV,HappyHour,RestaurantsGoodForGroups,GoodForMeal.breakfast,GoodForMeal.brunch,GoodForMeal.latenight,GoodForMeal.dinner,RestaurantsTableService,NoiseLevel,stars
pc1,0.13,-0.04,0.32,0.14,0.27,0.15,0.30,0.10,0.37,0.24,0.08,0.18,0.14,0.42,0.42,0.09,0.24
pc2,-0.07,-0.02,-0.08,0.18,0.07,-0.08,0.13,-0.25,-0.22,-0.08,0.63,0.59,-0.03,-0.20,-0.01,-0.08,0.15
pc3,0.40,-0.07,0.34,0.13,0.23,0.36,-0.36,-0.36,-0.06,-0.23,-0.12,-0.07,-0.23,-0.12,-0.02,-0.24,0.24
pc4,0.14,0.20,0.10,0.11,0.20,0.24,-0.35,0.16,0.19,0.00,0.16,0.17,0.24,-0.25,-0.09,0.50,-0.44
pc5,-0.26,0.16,-0.16,0.61,0.41,-0.29,0.04,-0.20,0.06,-0.19,-0.21,-0.21,0.18,-0.01,-0.12,0.13,0.16
pc6,0.05,0.77,0.06,-0.20,-0.23,0.03,-0.06,-0.02,-0.03,-0.23,0.04,0.01,0.38,0.08,0.06,-0.12,0.29
pc7,0.31,-0.29,-0.06,-0.02,-0.12,0.13,0.23,-0.38,-0.23,0.08,-0.01,-0.09,0.63,0.17,-0.27,0.08,-0.10
pc8,0.25,0.35,-0.20,0.30,0.07,0.19,0.10,0.19,-0.18,0.64,-0.01,-0.04,-0.14,-0.07,-0.26,-0.24,0.02
pc9,0.18,0.22,0.01,-0.10,-0.16,-0.10,0.18,-0.39,-0.07,0.11,-0.04,-0.05,-0.44,0.07,-0.03,0.67,0.15
pc10,-0.31,-0.03,-0.21,0.06,-0.02,0.74,0.27,0.19,-0.09,-0.32,-0.01,-0.03,-0.10,0.09,-0.14,0.17,0.15


---
## Step 3 — Weight matrix W

Shape: `(6 Spotify PCs × 13 Yelp PCs)`

Each row is a Spotify PC. Each column is a Yelp PC.
A positive value means: when a restaurant scores high on that Yelp PC, push toward that Spotify PC.
A negative value pulls away from it.

**Update these weights after reading the loadings above.**

In [4]:
N_SPOTIFY = len(SPOTIFY_PC_COLS)   # 6
N_YELP    = len(YELP_PC_COLS)      # 13

#                  yPC1   yPC2   yPC3   yPC4   yPC5   yPC6   yPC7   yPC8   yPC9  yPC10  yPC11  yPC12  yPC13
W = np.array([
    # sPC1          casual/energetic axis
    [ 0.8,  -0.3,   0.2,   0.1,   0.0,   0.0,   0.0,   0.0,   0.1,   0.0,   0.0,   0.0,   0.0],
    # sPC2
    [-0.5,   0.6,  -0.2,   0.2,   0.1,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0],
    # sPC3
    [ 0.2,   0.1,   0.7,  -0.3,   0.0,   0.1,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0],
    # sPC4
    [ 0.1,  -0.2,   0.1,   0.6,  -0.2,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0],
    # sPC5
    [ 0.0,   0.1,  -0.1,   0.1,   0.5,   0.2,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0],
    # sPC6
    [ 0.0,   0.0,   0.1,  -0.1,   0.2,   0.4,   0.1,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0],
], dtype=float)   # shape: (6, 13)

assert W.shape == (N_SPOTIFY, N_YELP), f"W must be ({N_SPOTIFY}, {N_YELP}), got {W.shape}"

W_df = pd.DataFrame(W, index=SPOTIFY_PC_COLS, columns=YELP_PC_COLS)
print("Weight matrix W  (rows=Spotify PCs, columns=Yelp PCs):")
display(W_df)

Weight matrix W  (rows=Spotify PCs, columns=Yelp PCs):


,pc1,pc2,pc3,pc4,pc5,pc6,pc7,pc8,pc9,pc10,pc11,pc12,pc13
pc1,0.8,-0.3,0.2,0.1,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0
pc2,-0.5,0.6,-0.2,0.2,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pc3,0.2,0.1,0.7,-0.3,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pc4,0.1,-0.2,0.1,0.6,-0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pc5,0.0,0.1,-0.1,0.1,0.5,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pc6,0.0,0.0,0.1,-0.1,0.2,0.4,0.1,0.0,0.0,0.0,0.0,0.0,0.0


---
## Step 4 — Trial run

In [ ]:
idx = 0
restaurant_row = restaurants.iloc[idx]

# ── STEP 1: raw Yelp features ─────────────────────────────────────────────
yelp_raw  = pd.read_csv(os.path.join(PROCESSED, 'yelp_ml_ready.csv'))
yelp_match = yelp_raw[yelp_raw['business_id'] == restaurant_row['business_id']]
print(f"Restaurant: {restaurant_row['name']}")
print("\n── STEP 1: Raw Yelp features ──")
display(yelp_match[YELP_FEATURES].T.rename(columns={yelp_match.index[0]: 'value'}))

# ── STEP 2: Yelp PC scores ────────────────────────────────────────────────
y = restaurant_vecs[idx]   # already computed by PCA sandbox
print("\n── STEP 2: Yelp PC scores (what PCA compressed the features into) ──")
display(pd.Series(y.round(3), index=YELP_PC_COLS).to_frame('score'))

# ── STEP 3: project into Spotify space via W ──────────────────────────────
s_hat = W @ y
print("\n── STEP 3: Target Spotify PC scores  (s_hat = W @ y) ──")
display(pd.Series(s_hat.round(3), index=SPOTIFY_PC_COLS).to_frame('target'))

# ── STEP 4: what does s_hat mean in audio terms? ─────────────────────────
print("\n── STEP 4: What that target means in audio terms ──")
audio_target = pd.Series(
    pca_spotify.inverse_transform(s_hat.reshape(1, -1))[0],
    index=AUDIO_FEATURES
).round(3)
display(audio_target.to_frame('audio value (standardised)'))

# ── STEP 5: cosine similarity → top 10 songs ─────────────────────────────
scores  = cosine_similarity(s_hat.reshape(1, -1), song_vecs)[0]
top_idx = np.argsort(scores)[::-1][:10]
results = songs.iloc[top_idx][['name', 'artists']].copy()
results.insert(0, 'score', scores[top_idx].round(3))
print("\n── STEP 5: Top 10 recommended songs ──")
display(results)

In [ ]:
import os

PROCESSED = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\processed"
yelp_raw  = pd.read_csv(os.path.join(PROCESSED, 'yelp_ml_ready.csv'))

# pick a restaurant to test
idx = 0
restaurant_row = restaurants.iloc[idx]
print(f"Restaurant: {restaurant_row['name']}\n")

# show its raw feature ratings
yelp_match   = yelp_raw[yelp_raw['business_id'] == restaurant_row['business_id']]
feature_vals = yelp_match[YELP_FEATURES].T
feature_vals.columns = ['value']
print("Feature profile:")
display(feature_vals)

# step 1 — grab its Yelp PC vector
y = restaurant_vecs[idx]                                          # shape (13,)

# step 2 — project into Spotify PC space
s_hat = W @ y                                                     # shape (6,)

# step 3 — cosine similarity against every song
scores = cosine_similarity(s_hat.reshape(1, -1), song_vecs)[0]   # shape (1.2M,)

# step 4 — top 10 results
top_idx = np.argsort(scores)[::-1][:10]
results  = songs.iloc[top_idx][['name', 'artists']].copy()
results.insert(0, 'score', scores[top_idx].round(3))

print("\nTop 10 recommended songs:")
display(results)